# Activity 5 — Time Series: Pull the Signal Apart

**DSC 402/502 Time & Trends.**

In [ ]:
import pandas as pd, numpy as np, statsmodels.api as sm, altair as alt
from statsmodels.tsa.seasonal import seasonal_decompose, STL

# ---------- a MENU of real series, all bundled / offline ----------
SERIES = {}
SERIES['CO2 (monthly, ppm)'] = sm.datasets.co2.load_pandas().data['co2'].resample('MS').mean().interpolate()
ap_csv = '112,118,132,129,121,135,148,148,136,119,104,118,115,126,141,135,125,149,170,170,158,133,114,140,145,150,178,163,172,178,199,199,184,162,146,166,171,180,193,181,183,218,230,242,209,191,172,194,196,196,236,235,229,243,264,272,237,211,180,201,204,188,235,227,234,264,302,293,259,229,203,229,242,233,267,269,270,315,364,347,312,274,237,278,284,277,317,313,318,374,413,405,355,306,271,306,315,301,356,348,355,422,465,467,404,347,305,336,340,318,362,348,363,435,491,505,404,359,310,337,360,342,406,396,420,472,548,559,463,407,362,405,417,391,419,461,472,535,622,606,508,461,390,432'
ap = [int(x) for x in ap_csv.split(',')]
SERIES['AirPassengers (monthly)'] = pd.Series(ap, index=pd.date_range('1949-01-01', periods=len(ap), freq='MS'))
ss = sm.datasets.sunspots.load_pandas().data
SERIES['Sunspots (annual)'] = pd.Series(ss['SUNACTIVITY'].values,
    index=pd.PeriodIndex(ss['YEAR'].astype(int).astype(str), freq='Y').to_timestamp())
en = sm.datasets.elnino.load_pandas().data.set_index('YEAR'); en.columns = range(1,13)
ens = en.stack(); ens.index = pd.to_datetime([f'{int(y)}-{int(m)}-01' for y,m in ens.index])
SERIES['El Nino SST (monthly)'] = ens.sort_index()

# ---------- one knob-box: classic (add/mult) OR STL (smoothness/robust) ----------
def decompose(series, method='classic', model='additive', period=12, seasonal=7, robust=False):
    if method=='stl':
        return STL(series, period=period, seasonal=seasonal, robust=robust).fit()
    return seasonal_decompose(series, model=model, period=period)

# ---------- turn any decomposition into 4 Altair panels ----------
def decomp_panels(result, title=''):
    tidy = pd.concat({'Observed':result.observed,'Trend':result.trend,
                      'Seasonal':result.seasonal,'Residual':result.resid}, axis=1)
    tidy.index.name='date'
    tidy = tidy.reset_index().melt(id_vars='date', var_name='component', value_name='value').dropna()
    base = alt.Chart(tidy).mark_line().encode(x=alt.X('date:T',title=None), y=alt.Y('value:Q',title=None)
        ).properties(height=85, width=560)
    return base.facet(row=alt.Row('component:N', sort=['Observed','Trend','Seasonal','Residual'],
        title=None)).resolve_scale(y='independent').properties(title=title)

print('Series on the menu:')
for k in SERIES:
    print('  -', k)

Series on the menu:
  - CO2 (monthly, ppm)
  - AirPassengers (monthly)
  - Sunspots (annual)
  - El Nino SST (monthly)


### 1. See it — the Keeling curve
This is the CO2 curve from lecture, split into Observed = Trend + Seasonal + Residual (additive, 12-month cycle). The trend you couldn't read off the raw wiggle is now a clean line; the annual “breathing” is its own panel.

In [ ]:
decomp_panels(decompose(SERIES['CO2 (monthly, ppm)'], method='classic', model='additive', period=12),
              'CO2 — additive, period=12')

alt.FacetChart(...)

### 2. Read the residual
If a decomposition fits, the **Residual** panel should look like structureless noise. If you still see waves or a shape in the residual, your choices haven't captured something. Keep an eye on it as you turn knobs below.

### 3. The knobs
- **`model`** — `additive` (Trend **+** Seasonal) if the swing stays the same size; `multiplicative` (Trend **×** Seasonal) if the swing **grows** with the level. *(classic only)*
- **`period`** — the cycle length. 12 for monthly; but e.g. **Sunspots** cycle is ~11 years and irregular — what you pick is a real call.
- **`method`** — `classic` (moving-average) or **`stl`** (Loess-based, the smoother from this week). STL adds two knobs:
- **`seasonal`** *(STL)* — bigger = a smoother, more stable seasonal shape; smaller = lets the seasonal pattern change over time.
- **`robust`** *(STL)* — `True` down-weights outliers/spikes (useful for messy real series like El Niño).

Note: STL = Seasonal-Trend decomposition using Loess

Quick demo — same CO2 series, STL + robust, smoother seasonal. Notice it's a *different* (but also defensible) picture:

In [ ]:
decomp_panels(decompose(SERIES['CO2 (monthly, ppm)'], method='stl', period=12, seasonal=13, robust=True),
              'CO2 — STL, seasonal=13, robust')

alt.FacetChart(...)

### 4. Your turn — pick a series, tune the knobs
Pick **any** series from the menu and tune the knobs until you get a decomposition **you would defend**. The series are all different, so your choices (and your findings) may differ from your classmates':
- **CO2** — steady trend + steady cycle
- **AirPassengers** — swing grows with the trend (hint: is it really additive?)
- **Sunspots** — irregular ~11-year cycle; **what `period` is even right?**
- **El Niño SST** — clean annual cycle *plus* irregular multi-year pattern


In [ ]:
print(list(SERIES))

['CO2 (monthly, ppm)', 'AirPassengers (monthly)', 'Sunspots (annual)', 'El Nino SST (monthly)']


In [ ]:
# ---- EDIT these, re-run, and compare. There is no single right setting. ----
name   = 'Sunspots (annual)'   # pick from SERIES (print(list(SERIES)) to see them)
series = SERIES[name]

result = decompose(series,
    method   = 'stl',    # 'classic' or 'stl'
    model    = 'additive',   # 'additive' or 'multiplicative'   (classic only)
    period   = 11,           # 12 monthly; try 11 for sunspots; experiment
    seasonal = 9,            # STL only: larger = smoother seasonal
    robust   = False,        # STL only: True down-weights spikes
)
decomp_panels(result, name)

alt.FacetChart(...)

### 5. Compare and decide
Try at least **two** different settings on your series (e.g. classic-multiplicative vs STL-robust, or two periods). Keep the one whose **residual** looks most like noise *and* whose trend/seasonal you'd actually stand behind. Copy the cell above and change the knobs, or edit and re-run.

In [ ]:
# YOUR TURN: a second configuration to compare against the first

name   = 'Sunspots (annual)'   # pick from SERIES (print(list(SERIES)) to see them)
series = SERIES[name]

result = decompose(series,
    method   = 'classic',    # 'classic' or 'stl'
    model    = 'additive',   # 'additive' or 'multiplicative'   (classic only)
    period   = 11,           # 12 monthly; try 11 for sunspots; experiment
    seasonal = 9,            # STL only: larger = smoother seasonal
    robust   = False,        # STL only: True down-weights spikes
)
decomp_panels(result, name)

alt.FacetChart(...)

### 6. Decide + defend
Write **2–3 sentences:** which series you chose, the **knob settings** you landed on and **why** (what the residual told you, why that period/model/smoothing), and **one thing your decomposition revealed that the raw curve hid.** Different defensible choices are fine — the defense is the grade.

---
### Submit
🎧 **Post to `#05-time-series` for credit.**

Post **your decomposition + your 2–3 sentence defense** (series, knob settings + why, one hidden thing revealed).